<a href="https://colab.research.google.com/github/exdsgift/NerGuard/blob/main/PII_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True

Setup & import

In [ ]:
!pip install -q "transformers>=4.40" datasets seqeval accelerate

import torch
import os, json, unicodedata
import numpy as np
from typing import List, Dict, Tuple
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from seqeval.metrics import f1_score, classification_report
from seqeval.scheme import IOB2
import pprint as pp

In [ ]:
from datasets import load_dataset, DatasetDict

ds = load_dataset("gretelai/synthetic_pii_finance_multilingual")
print(ds)

In [ ]:
if "validation" not in ds:
    tmp = ds["test"].train_test_split(test_size=0.5, seed=42)
    ds = DatasetDict({
        "train": ds["train"],
        "validation": tmp["test"],
        "test": tmp["train"]
    })
print(ds)

In [ ]:
ds.shape

In [ ]:
pp.pprint(ds['train'][0])

Tokenizer (fast) and support functions

In [ ]:
tok = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base", use_fast=True)
assert tok.is_fast, "Serve tokenizer fast per offset_mapping"

In [ ]:
def parse_spans(spans):
    if isinstance(spans, str):
        try:
            spans = json.loads(spans)
        except json.JSONDecodeError:
            return []
    if not isinstance(spans, list):
        return []
    out = []
    for s in spans:
        if isinstance(s, dict) and all(k in s for k in ("start","end","label")):
            out.append(s)
    return out

In [ ]:
def collect_label_set(dataset_dict):
    labels = set()
    for split in dataset_dict.keys():
        for ex in dataset_dict[split]:
            for s in parse_spans(ex["pii_spans"]):
                labels.add(s["label"])
    return sorted(labels)

entity_labels = collect_label_set(ds)  # es: ['company','date','name','street_address', ...]
bio_labels = ["O"] + [f"B-{l}" for l in entity_labels] + [f"I-{l}" for l in entity_labels]
label2id = {l:i for i,l in enumerate(bio_labels)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
bio_labels

spans → BIO with offset_mapping

In [ ]:
def spans_to_bio(text, spans, label2id, tokenizer, max_length=256):
    import unicodedata
    text = unicodedata.normalize("NFC", text)
    spans = parse_spans(spans)

    # 1) maschera char-level
    char_tags = ["O"] * len(text)
    for s in spans:
        start, end, lab = s.get("start"), s.get("end"), s.get("label")
        if not isinstance(start,int) or not isinstance(end,int) or not isinstance(lab,str):
            continue
        if start < 0 or end <= start or end > len(text):
            continue
        if f"B-{lab}" not in label2id or f"I-{lab}" not in label2id:
            continue
        char_tags[start] = f"B-{lab}"
        for i in range(start+1, end):
            char_tags[i] = f"I-{lab}"

    # 2) tokenizza con special e offsets
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_attention_mask=True,
        add_special_tokens=True,     # <-- importante
        truncation=True,
        max_length=max_length
    )
    offsets = enc["offset_mapping"]

    tags = []
    for (a, b) in offsets:
        if a == 0 and b == 0:
            tags.append(None)        # special token -> ignora in loss
            continue
        if a == b:
            tags.append("O")
            continue
        start_tag = char_tags[a] if 0 <= a < len(text) else "O"
        if start_tag.startswith("B-"):
            lab = start_tag
        else:
            window = [t for t in char_tags[a:b] if t != "O"]
            if window:
                any_lab = window[0]
                lab = "I-" + any_lab.split("-", 1)[1]
            else:
                lab = "O"
        tags.append(lab)

    ignore_index = -100
    label_ids = [ignore_index if t is None else label2id[t] for t in tags]
    return enc["input_ids"], enc["attention_mask"], label_ids


Processing all splits

In [ ]:
TEXT_COL = "generated_text"
SPAN_COL = "pii_spans"

def preprocess_batch(batch):
    input_ids, attention_mask, labels = [], [], []
    for text, spans in zip(batch[TEXT_COL], batch[SPAN_COL]):
        ids, mask, y = spans_to_bio(text, spans, label2id, tok, max_length=512)
        input_ids.append(ids); attention_mask.append(mask); labels.append(y)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

remove_cols = list(ds["train"].column_names)
processed = DatasetDict()
for split in ds.keys():
    processed[split] = ds[split].map(
        preprocess_batch,
        batched=True,
        remove_columns=remove_cols
    ).with_format("torch",
                  columns=["input_ids","attention_mask"],
                  output_all_columns=True)

#.with_format("torch", columns=["input_ids","attention_mask","labels"])
train_processed = processed["train"]
val_processed   = processed["validation"]
test_processed  = processed.get("test", None)

In [ ]:
from pprint import pprint

print("num_labels:", len(label2id))
pprint(label2id)           # label string -> id
# If you also want the reverse:
id2label = {i:l for l,i in label2id.items()}


In [ ]:
# take one example from the processed train set
ex = train_processed[0]

tokens = tok.convert_ids_to_tokens(ex["input_ids"])
tag_ids = ex["labels"]
tag_str = [id2label[int(i)] for i in tag_ids if i != -100]
aligned_data = [(tokens[i], tag_ids[i], id2label[int(tag_ids[i])]) for i in range(len(tokens)) if tag_ids[i] != -100]

for t, tid, ts in aligned_data:
    print(f"{t:20s}  {tid:3d}  {ts}")

In [ ]:
from collections import Counter

used_ids = Counter()
for example in train_processed:
    used_ids.update(i for i in example["labels"] if i != -100)   # ignore masked

print("Unique label IDs used:", len(used_ids))
for lid, cnt in used_ids.most_common()[:30]:
    print(f"{lid:3d}  {id2label[lid]:30s}  {cnt}")

Collator (padding dinamico + pad label = -100), Per softmax (senza CRF) usiamo -100, che la CrossEntropy ignora.

In [ ]:
# @title
class NERCollatorSoftmax:
    def __init__(self, tokenizer, label_pad_id=-100, pad_to_multiple_of=32):
        self.inner = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=pad_to_multiple_of)
        self.label_pad_id = label_pad_id
    def __call__(self, features):
        labels = [f["labels"] for f in features]
        inputs = [{k:v for k,v in f.items() if k!="labels"} for f in features]
        batch = self.inner(inputs)
        max_len = batch["input_ids"].shape[1]
        padded = [y + [self.label_pad_id]*(max_len-len(y)) for y in labels]
        batch["labels"] = torch.tensor(padded, dtype=torch.long)
        return batch

collator = NERCollatorSoftmax(tok)

In [ ]:
!pip install -q evaluate

In [ ]:
from transformers import DataCollatorForTokenClassification
import evaluate
from datasets import load_dataset

collator = DataCollatorForTokenClassification(tok)
metric = evaluate.load("seqeval")

Modello (senza CRF) + metriche

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/mdeberta-v3-base",
    num_labels=len(bio_labels),
    id2label=id2label,
    label2id=label2id
)

Metriche (argmax su logits):

In [ ]:
# This method is not optimal because assumes all -100 are at the end.
# I mask non-first subwords with -100, those can appear in the middle, so you’ll misalign y/p.
# Instead, filter by the mask position-wise.

def compute_metrics_softmax(eval_pred):
    pred_logits, label_ids = eval_pred   # [B,T,K], [B,T]
    pred_ids = pred_logits.argmax(-1)

    y_true, y_pred = [], []
    for yt, yp in zip(label_ids, pred_ids):
        true_tags = [id2label[int(i)] for i in yt if int(i) != -100]
        pred_tags = [id2label[int(i)] for i in yp[:len(true_tags)]]
        y_true.append(true_tags); y_pred.append(pred_tags)

    f1 = f1_score(y_true, y_pred, scheme=IOB2, mode="strict")
    print(classification_report(y_true, y_pred, scheme=IOB2, mode="strict"))
    return {"f1": f1}


In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from seqeval.scheme import IOB2  # or IOBES if you use BIOES

def compute_metrics_softmax(eval_pred):
    logits, labels = eval_pred             # logits: [B,T,K], labels: [B,T]
    pred_ids = logits.argmax(-1)           # [B,T]

    y_true, y_pred = [], []
    for yt, yp in zip(labels, pred_ids):
        true_seq, pred_seq = [], []
        for tid, pid in zip(yt, yp):
            tid = int(tid)
            if tid == -100:                # skip masked positions (PAD, CLS/SEP, non-first subwords, etc.)
                continue
            true_seq.append(id2label[tid])
            pred_seq.append(id2label[int(pid)])
        y_true.append(true_seq)
        y_pred.append(pred_seq)

    # entity-level (strict) scores
    p = precision_score(y_true, y_pred, scheme=IOB2, mode="strict")
    r = recall_score(y_true, y_pred, scheme=IOB2, mode="strict")
    f1 = f1_score(y_true, y_pred, scheme=IOB2, mode="strict")

    # Optional (nice for debugging, but can spam logs during eval)
    # print(classification_report(y_true, y_pred, scheme=IOB2, mode="strict"))

    return {"precision": p, "recall": r, "f1": f1}


Training Arguments (compat con versioni diverse)

In [ ]:
from transformers import TrainingArguments
import transformers

args = TrainingArguments(
    output_dir="pii_mdeberta_softmax",
    per_device_train_batch_size=2,      # ↓ batch minimo
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,     # ↑ batch effettivo = 32
    learning_rate=2e-5,
    num_train_epochs=1,                 # 1 epoch for smoke test, keep it high
    eval_strategy= "epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=True,                          # mixed precision su T4
    gradient_checkpointing=False,       # True di solito ma non funziona su colab
    dataloader_num_workers=2,
    dataloader_pin_memory=False,        # riduce picchi
    eval_accumulation_steps=1,          # no buffer gigante in eval
    logging_steps=50,
    logging_strategy="steps",
    report_to=[],
    save_safetensors=True
)

### Training

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_processed,
    eval_dataset=val_processed,
    data_collator=collator,
    processing_class=tok,
    compute_metrics=compute_metrics_softmax
)

In [ ]:
!nvidia-smi

In [ ]:
trainer.train()

trainer.save_model("/content/drive/MyDrive/Colab data/content/pii_mdeberta_softmax/outputs/best")
tok.save_pretrained("/content/drive/MyDrive/Colab data/content/pii_mdeberta_softmax/outputs/best")

### Metrics

In [ ]:
import pprint as pp

# val set
val_metrics = trainer.evaluate()
print("val metrics")
pp.pprint(val_metrics)

# test set
print()
print("test metrics")
test_metrics = trainer.evaluate(test_processed)
pp.pprint(test_metrics)

### Graphs

In [ ]:
import pandas as pd

history = trainer.state.log_history
df = pd.DataFrame(history)

# colonne tipiche: 'loss', 'learning_rate', 'epoch', 'step',
#                  'eval_loss', 'eval_f1', 'eval_precision', 'eval_recall', ...
df.head()

In [ ]:
import matplotlib.pyplot as plt

# Training loss vs step
df_tr = df[df["loss"].notna()]
plt.figure()
plt.plot(df_tr["step"], df_tr["loss"])
plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title("Training loss"); plt.grid(True)
plt.show()

# Eval F1 / Precision / Recall vs step (se fai evaluation per step)
df_ev = df[df["eval_f1"].notna()]
plt.figure()
plt.plot(df_ev["step"], df_ev["eval_f1"], label="F1")
if "eval_precision" in df_ev: plt.plot(df_ev["step"], df_ev["eval_precision"], label="Precision")
if "eval_recall" in df_ev:    plt.plot(df_ev["step"], df_ev["eval_recall"],    label="Recall")
plt.xlabel("Step"); plt.ylabel("Score"); plt.title("Eval metrics"); plt.legend(); plt.grid(True)
plt.show()

# Eval loss vs step
if "eval_loss" in df:
    df_ev_loss = df[df["eval_loss"].notna()]
    plt.figure()
    plt.plot(df_ev_loss["step"], df_ev_loss["eval_loss"])
    plt.xlabel("Step"); plt.ylabel("Eval loss"); plt.title("Eval loss"); plt.grid(True)
    plt.show()


In [ ]:
import numpy as np

df_tr = df[df["loss"].notna()].copy()
df_tr["loss_ma"] = df_tr["loss"].rolling(window=20, min_periods=1).mean()

plt.figure()
plt.plot(df_tr["step"], df_tr["loss"], alpha=0.3, label="raw")
plt.plot(df_tr["step"], df_tr["loss_ma"], label="moving avg (20)")
plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title("Training loss (smoothed)"); plt.legend(); plt.grid(True)
plt.show()


In [ ]:
by_epoch = df.groupby("epoch", dropna=True).agg({
    "loss":"last",
    "eval_loss":"last",
    "eval_f1":"max",
    "eval_precision":"max",
    "eval_recall":"max"
})
print(by_epoch)

plt.figure()
if "eval_f1" in by_epoch: plt.plot(by_epoch.index, by_epoch["eval_f1"], marker="o")
plt.xlabel("Epoch"); plt.ylabel("F1 (best per epoch)"); plt.title("Eval F1 per epoch"); plt.grid(True)
plt.show()


### Tester

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "outputs/best"
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

from transformers import pipeline
clf = pipeline("text-classification",
               model=model,
               tokenizer=tok,
               device_map="auto")
text_sample = "Mi chiamo Gabriele Durante, vivo a Milano. Email: g.durante@example.com, Carta di credito: 4111 1111 1111 1111; tel: +39 345 1234567"
clf(text_sample, top_k=None)